In [ ]:
import jax.numpy as jnp
import matplotlib.pyplot as plt
import jax
from jax.scipy.special import bessel_jn
from fft import *
import numpy as np
from params import *

jax.config.update("jax_enable_x64", True)
jax.devices()
N = 257

In [ ]:
W = p
H = p
plot_unit_l = unit_l * 1e6  # um
r_extent = [
    -W / 2 * plot_unit_l,
    W / 2 * plot_unit_l,
    -H / 2 * plot_unit_l,
    H / 2 * plot_unit_l,
]
X, Y, R = create_axes(W, H, 1, N=N)
# F_airy = fft2_shifted(E_airy)
f_x, f_y = ft_image_coordinates(X, W, H)
k_x = 2 * jnp.pi * f_x
k_y = 2 * jnp.pi * f_y
k = jnp.sqrt(k_x**2 + k_y**2)

In [ ]:
@jit
def make_circular_mask(R, radius):
    return jnp.astype(R <= radius, jnp.float64)


@jit
def calc_power(x):
    return jnp.sum(jnp.abs(x) ** 2)


@jit
def make_phi_design(arg_phase, phase_round_off):
    phi_rounded = jnp.floor((arg_phase + phase_round_off) / jnp.pi + 1.5)
    phi_design = jnp.pi * jnp.mod(phi_rounded, 2)
    return phi_design


def fractional_disk_mask(X, Y, R, dx, dy, m=4):
    # X,Y are cell centers, shape (Ny, Nx)
    # subcell offsets in [-0.5, 0.5]
    ox = (jnp.arange(m) + 0.5) / m - 0.5
    oy = (jnp.arange(m) + 0.5) / m - 0.5
    # broadcast to (m,m,Ny,Nx)
    Xs = X[None, None, :, :] + ox[:, None, None, None] * dx
    Ys = Y[None, None, :, :] + oy[None, :, None, None] * dy
    inside = (Xs**2 + Ys**2) <= R**2
    # print(inside)
    return inside.mean(axis=(0, 1))  # (Ny,Nx) in [0,1]


# F_airy = make_circular_mask(k, k_0 * NA)
dk_x = k_x[0, 1] - k_x[0, 0]
dk_y = dk_x
# F_airy = fractional_disk_mask(k_x,k_y, k_0 * NA, dk_x,dk_y,m=8)
F_airy = make_circular_mask(k, k_0 * NA)

# k_filter = fractional_disk_mask(k_x,k_y, k_0, dk_x,dk_y,m=8)
k_filter = make_circular_mask(k, k_0)

R_airy = 0.61 * lambda_0 / (NA * unit_l)
# R_mask = make_circular_mask(R, R_airy)

dx = X[0, 1] - X[0, 0]
dy = dx
# R_mask = fractional_disk_mask(X, Y, R_airy, dx,dy, m=8)
R_mask = make_circular_mask(R, R_airy)


@jit
def calc_phi_prop(d=d, d_cov=d_cov):
    return d * jnp.sqrt(k_0**2 - k_filter * k**2 + eps) + d_cov * jnp.sqrt(
        (n_cov * k_0) ** 2 - k_filter * k**2 + eps
    )


@jit
def calculate_Q_relative(pattern):
    total_P = calc_power(pattern)
    Q_p = jnp.sum(jnp.abs(pattern) ** 2 * R_mask)
    # print(Q_p,total_P)
    return Q_p / total_P


@jit
def backward_pass(F_airy, d=d, d_cov=d_cov):
    phi_prop = calc_phi_prop(d, d_cov)
    F_exit = jnp.exp(-1j * phi_prop) * F_airy * k_filter
    E_exit = ifft2_shifted(F_exit)
    return E_exit


@jit
def forward_pass(T_design, d=d, d_cov=d_cov):
    T_spectrum = fft2_shifted(T_design)
    phi_prop = calc_phi_prop(d, d_cov)
    F_im = jnp.exp(1j * phi_prop) * T_spectrum * k_filter
    E_im = ifft2_shifted(F_im)
    return E_im


plt.imshow(R_mask)

In [ ]:
def plot_real(x, cmap="hot"):
    fig, ax = plt.subplots()
    ax.imshow(x, extent=r_extent, cmap=cmap)
    # plt
    # ax.set_aspect('auto') #
    # ax.set_box_aspect(1)
    plt.xlabel(r"$x$ [um]")
    plt.ylabel(r"$y$ [um]")
    plt.show()

In [ ]:
# phase_round_off = 0
# phase_round_off = -0.48611 * jnp.pi
phase_round_off = 0.39 * jnp.pi
d_val = d * 1.0
E_exit = backward_pass(F_airy, d=d_val)
arg_phase = jnp.angle(E_exit)
plot_real(jnp.abs(E_exit))

phi_design = make_phi_design(arg_phase, phase_round_off)
T_design = 1 / N * jnp.exp(1j * phi_design)
plot_real(jnp.angle(-T_design), cmap="binary")

E_im = forward_pass(T_design, d=d_val)
plot_real(jnp.abs(E_im))
plot_real(jnp.abs(E_im) ** 2)
calculate_Q_relative(E_im)

In [ ]:
d_vals = jnp.linspace(525e-6 / unit_l, 545e-6 / unit_l, 200)
epsilons = jnp.linspace(0, jnp.pi, 100)
Qs = np.zeros((len(d_vals), len(epsilons)))
# Qs
for i, d_val in enumerate(d_vals):
    E_exit = backward_pass(F_airy, d=d_val)
    arg_phase = jnp.angle(E_exit)
    for j, phase_round_off in enumerate(epsilons):
        phi_design = make_phi_design(arg_phase, phase_round_off)
        T_design = 1 / N * jnp.exp(1j * phi_design)
        E_im = forward_pass(T_design, d=d_val)
        Q = calculate_Q_relative(E_im)
        Qs[i, j] = Q
        # print(Q)

## $Q$ as function of $d$ and phase round off $\epsilon$
TODO: investigate reason(s) for difference between this and paper result

In [ ]:
fig, ax = plt.subplots()
im = ax.imshow(
    jnp.transpose(Qs),
    label=r"$Q$",
    extent=[
        jnp.min(d_vals) * plot_unit_l,
        jnp.max(d_vals) * plot_unit_l,
        jnp.min(epsilons) / jnp.pi,
        jnp.max(epsilons) / jnp.pi,
    ],
    aspect="auto",
)
cb = fig.colorbar(im, ax=ax)
ax.set_box_aspect(1)
ax.set_xlabel(r"$d$ [um]")
ax.set_ylabel(r"$\epsilon/\pi$")